In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, SimpleRNN, Dropout

data_path = r"D:\lecture\PTIT\20251\IntSys\weather-vn"

file_list = [
    "df_weather.csv",
    "precipitation_mean_max_min_temp_1901_2020_en.csv",
    "weather.csv"
]

dfs = []
for f in file_list:
    df = pd.read_csv(os.path.join(data_path, f))
    print("File:", f)
    print(df.head(), "\n")
    dfs.append(df)

cleaned_dfs = []

for df in dfs:
    # chỉ lấy cột số
    df_num = df.select_dtypes(include=['number']).copy()
    
    # nếu không có dữ liệu số → bỏ qua
    if df_num.shape[1] == 0:
        continue
        
    # bỏ NaN
    df_num = df_num.fillna(method='ffill').fillna(method='bfill')
    
    cleaned_dfs.append(df_num)

print("Số dataset hợp lệ:", len(cleaned_dfs))

TARGET_DIM = 5
pca = PCA(n_components=TARGET_DIM)

merged_data = []

for df in cleaned_dfs:
    # chuẩn hóa từng dataset riêng
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df)
    
    # chuyển về không gian PCA chung
    reduced = pca.fit_transform(scaled)
    
    merged_data.append(reduced)

# gộp lại thành 1 dataset
final_data = np.vstack(merged_data)

print("Final merged dataset shape:", final_data.shape)

def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length, 0])  # dự báo feature đầu tiên
    return np.array(X), np.array(y)

seq_length = 30
X, y = create_sequences(final_data, seq_length)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

X_train.shape, y_train.shape


File: df_weather.csv
     location.name              location.region location.terrain  \
0         An Giang      Đồng Bằng Sông Cửu Long        đồng bằng   
1  Bà Rịa-Vũng Tàu                  Đông Nam Bộ         ven biển   
2        Bắc Giang  Trung du và miền núi Bắc Bộ         miền núi   
3          Bắc Kạn  Trung du và miền núi Bắc Bộ         miền núi   
4         Bạc Liêu      Đồng Bằng Sông Cửu Long         ven biển   

  location.country  location.lat  location.lon        date  date_epoch  \
0          Vietnam       10.7000      105.1167  2024-04-21  1713657600   
1          Vietnam       10.3500      107.0667  2024-04-21  1713657600   
2          Vietnam       21.2667      106.2000  2024-04-21  1713657600   
3          Vietnam       22.1333      105.8333  2024-04-21  1713657600   
4          Vietnam        9.2850      105.7244  2024-04-21  1713657600   

   day.maxtemp_c  day.maxtemp_f  ...             day.condition.text  \
0           38.6          101.5  ...                  

C:\Users\Admin\AppData\Local\Temp\ipykernel_1396\524357439.py:40: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_num = df_num.fillna(method='ffill').fillna(method='bfill')
C:\Users\Admin\AppData\Local\Temp\ipykernel_1396\524357439.py:40: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_num = df_num.fillna(method='ffill').fillna(method='bfill')
C:\Users\Admin\AppData\Local\Temp\ipykernel_1396\524357439.py:40: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_num = df_num.fillna(method='ffill').fillna(method='bfill')


((167157, 30, 5), (167157,))

In [ ]:
rnn_model = Sequential([
    SimpleRNN(64, return_sequences=True, input_shape=(seq_length, TARGET_DIM)),
    SimpleRNN(64, return_sequences=True),
    SimpleRNN(64, return_sequences=True),
    SimpleRNN(64, return_sequences=True),
    SimpleRNN(64, return_sequences=True),
    SimpleRNN(64, return_sequences=True),
    SimpleRNN(64),
    Dense(1)
])

rnn_model.compile(optimizer='adam', loss='mse')
rnn_model.summary()

history_rnn = rnn_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, y_test)
)

lstm_model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(seq_length, TARGET_DIM)),
    LSTM(64, return_sequences=True),
    LSTM(64, return_sequences=True),
    LSTM(64, return_sequences=True),
    LSTM(64, return_sequences=True),
    LSTM(64, return_sequences=True),
    LSTM(64),
    Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mse')
lstm_model.summary()

history_lstm = lstm_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, y_test)
)

pred_rnn = rnn_model.predict(X_test)
pred_lstm = lstm_model.predict(X_test)

plt.figure(figsize=(14,5))
plt.plot(y_test, label='Actual')
plt.plot(pred_rnn, label='RNN Forecast')
plt.plot(pred_lstm, label='LSTM Forecast')
plt.legend()
plt.show()


C:\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ simple_rnn_7 (SimpleRNN)             │ (None, 30, 64)              │           4,480 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn_8 (SimpleRNN)             │ (None, 30, 64)              │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn_9 (SimpleRNN)             │ (None, 30, 64)              │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn_10 (SimpleRNN)            │ (None, 30, 64)              │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn_11 (SimpleRNN)            │ (None, 30, 64)              │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn_12 (SimpleRNN)            │ (None, 30, 64)              │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn_13 (SimpleRNN)            │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 54,081 (211.25 KB)

 Trainable params: 54,081 (211.25 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 30, 64)              │          17,920 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 30, 64)              │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_2 (LSTM)                        │ (None, 30, 64)              │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_3 (LSTM)                        │ (None, 30, 64)              │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_4 (LSTM)                        │ (None, 30, 64)              │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_5 (LSTM)                        │ (None, 30, 64)              │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_6 (LSTM)                        │ (None, 64)                  │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 216,129 (844.25 KB)

 Trainable params: 216,129 (844.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
 347/5224 ━━━━━━━━━━━━━━━━━━━━ 6:01 74ms/step - loss: 0.1035